# 제8장: 유즈케이스 발굴

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch08.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

AI 유즈케이스 발굴은 **기술 가능성**과 **거버넌스 적합성**을 동시에 평가해야 합니다.
이 장에서는 유즈케이스 적합성 평가, 비즈니스 케이스 정량화, 기술 성숙도 평가,
로드맵 우선순위화, 쉐도우 AI 탐지 도구를 구현합니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**AI 종합 적합성 점수 산출 엔진(Suitability Assessment Engine)**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum

class Decision(Enum):
    GO = "GO"
    CONDITIONAL_GO = "CONDITIONAL_GO"
    HOLD = "HOLD"
    NO_GO = "NO_GO"

@dataclass
class AssessmentDimension:
    """Individual assessment dimension with score and findings."""
    name: str
    score: float  # 0-100
    weight: float
    findings: List[str] = field(default_factory=list)
    blockers: List[str] = field(default_factory=list)

@dataclass
class SuitabilityResult:
    """Comprehensive suitability assessment result."""
    overall_score: float
    decision: Decision
    dimensions: List[AssessmentDimension]
    roi_estimate: Dict[str, float]
    recommendations: List[str]
    risk_flags: List[str]

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class AISuitabilityAssessment:
    """AI suitability assessment engine.
    Evaluates use cases across five dimensions:
    technical feasibility, business value, data readiness,
    organizational readiness, and governance risk.
    """
    WEIGHTS = {
        'technical': 0.30, 'business': 0.30,
        'data_readiness': 0.20, 'org_readiness': 0.10,
        'governance': 0.10
    }
    THRESHOLDS = {
        Decision.GO: 75, Decision.CONDITIONAL_GO: 60,
        Decision.HOLD: 40
    }

    def assess(self, use_case: Dict) -> SuitabilityResult:
        dims = [
            self._assess_technical(use_case),
            self._assess_business(use_case),
            self._assess_data_readiness(use_case),
            self._assess_org_readiness(use_case),
            self._assess_governance(use_case),
        ]
        overall = sum(d.score * d.weight for d in dims)
        blockers = [b for d in dims for b in d.blockers]

        if blockers:
            decision = Decision.NO_GO
        elif overall >= self.THRESHOLDS[Decision.GO]:
            decision = Decision.GO
        elif overall >= self.THRESHOLDS[Decision.CONDITIONAL_GO]:
            decision = Decision.CONDITIONAL_GO
        elif overall >= self.THRESHOLDS[Decision.HOLD]:
            decision = Decision.HOLD
        else:
            decision = Decision.NO_GO

        roi = self._estimate_roi(use_case)
        recs = self._generate_recommendations(dims)
        return SuitabilityResult(
            round(overall, 1), decision, dims,
            roi, recs, blockers
        )

    def _assess_technical(self, uc: Dict) -> AssessmentDimension:
        scores, findings, blockers = [], [], []
        ptype = uc.get('problem_type', 'unknown')
        if ptype in ['classification', 'regression']:
            scores.append(25)
        elif ptype in ['generation', 'reasoning']:
            scores.append(15)
            findings.append("Generative AI requires additional governance")
        else:
            scores.append(5)
        scores.append(min(uc.get('algorithm_maturity', 0), 25))
        scores.append(min(uc.get('data_volume_score', 0), 25))
        scores.append(min(uc.get('infrastructure_score', 0), 25))
        total = sum(scores)
        if total < 20:
            blockers.append("BLOCKER: Technical feasibility critically low")
        return AssessmentDimension(
            'technical', total, self.WEIGHTS['technical'],
            findings, blockers
        )

    def _assess_business(self, uc: Dict) -> AssessmentDimension:
        score = (uc.get('strategic_alignment', 0) * 0.4
                 + uc.get('urgency_score', 0) * 0.3
                 + uc.get('business_impact', 0) * 0.3)
        findings = []
        if score < 50:
            findings.append("Low business value - consider deprioritizing")
        return AssessmentDimension(
            'business', min(score, 100),
            self.WEIGHTS['business'], findings
        )

    def _assess_data_readiness(self, uc: Dict) -> AssessmentDimension:
        drl_map = {'A': 20, 'B': 40, 'C': 60, 'D': 80, 'E': 100}
        score = drl_map.get(uc.get('data_readiness_level', 'A'), 0)
        blockers = []
        if uc.get('has_pii') and not uc.get('pii_consent'):
            blockers.append("BLOCKER: PII without consent framework")
        return AssessmentDimension(
            'data_readiness', score,
            self.WEIGHTS['data_readiness'], [], blockers
        )

    def _assess_org_readiness(self, uc: Dict) -> AssessmentDimension:
        score = (uc.get('ai_talent_score', 0) * 0.3
                 + uc.get('process_maturity', 0) * 0.25
                 + uc.get('data_culture', 0) * 0.25
                 + uc.get('leadership_support', 0) * 0.2)
        return AssessmentDimension(
            'org_readiness', min(score, 100),
            self.WEIGHTS['org_readiness']
        )

    def _assess_governance(self, uc: Dict) -> AssessmentDimension:
        gov_score = 100 - uc.get('risk_level', 50)
        blockers = []
        if uc.get('prohibited_use'):
            blockers.append("BLOCKER: Prohibited AI category")
        if uc.get('high_risk') and not uc.get('impact_assessment_done'):
            blockers.append("BLOCKER: High-risk without impact assessment")
        return AssessmentDimension(
            'governance', gov_score,
            self.WEIGHTS['governance'], [], blockers
        )

    def _estimate_roi(self, uc: Dict) -> Dict[str, float]:
        benefit = uc.get('estimated_annual_benefit', 0)
        cost = uc.get('estimated_total_cost', 1)
        return {
            'optimistic': round((benefit * 1.3 - cost) / cost * 100, 1),
            'base': round((benefit - cost) / cost * 100, 1),
            'pessimistic': round((benefit * 0.5 - cost) / cost * 100, 1)
        }

    def _generate_recommendations(
        self, dims: List[AssessmentDimension]
    ) -> List[str]:
        recs = []
        for d in dims:
            if d.score < 40:
                recs.append(
                    f"[{d.name}] Significant improvement needed"
                )
            elif d.score < 60:
                recs.append(
                    f"[{d.name}] Moderate gaps - create remediation plan"
                )
        return recs

**유스케이스 등록 및 관리 시스템(Use Case Registry)**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum
from datetime import datetime
import uuid

class RiskGrade(Enum):
    PROHIBITED = 1  # 금지
    HIGH = 2        # 고위험
    LIMITED = 3     # 제한
    MINIMAL = 4     # 최소

class ProposalStatus(Enum):
    DRAFT = "DRAFT"
    SUBMITTED = "SUBMITTED"
    SCREENING = "SCREENING"
    COMMITTEE_REVIEW = "COMMITTEE_REVIEW"
    APPROVED = "APPROVED"
    CONDITIONAL = "CONDITIONAL"
    REJECTED = "REJECTED"
    ON_HOLD = "ON_HOLD"

@dataclass
class EthicsScreening:
    fairness: str = "unchecked"     # n/a, confirmed, unchecked, no_mitigation
    transparency: str = "unchecked"
    privacy: str = "unchecked"
    safety: str = "unchecked"
    accountability: str = "unchecked"
    explainability: str = "unchecked"
    environmental: str = "unchecked"
    dual_use: str = "unchecked"

    def has_blockers(self) -> bool:
        vals = [
            self.fairness, self.transparency, self.privacy,
            self.safety, self.accountability, self.explainability,
            self.environmental, self.dual_use
        ]
        return any(v in ("unchecked", "no_mitigation") for v in vals)

@dataclass
class UseCaseProposal:
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    title: str = ""
    department: str = ""
    proposer: str = ""
    background: str = ""
    objectives: str = ""
    ai_type: str = ""
    data_sources: List[str] = field(default_factory=list)
    target_users: str = ""
    estimated_budget: float = 0.0
    timeline_months: int = 0
    is_generative: bool = False
    status: ProposalStatus = ProposalStatus.DRAFT
    risk_grade: Optional[RiskGrade] = None
    ethics_screening: Optional[EthicsScreening] = None
    created_at: datetime = field(default_factory=datetime.now)
    updated_at: datetime = field(default_factory=datetime.now)

class RiskClassifier:
    """Automated risk classification based on EU AI Act and Korean AI Basic Act."""

    PROHIBITED_KEYWORDS = [
        "social_scoring", "subliminal_manipulation",
        "real_time_biometric_surveillance",
        "emotion_exploitation_vulnerable"
    ]
    HIGH_RISK_KEYWORDS = [
        "hiring", "credit_scoring", "medical_diagnosis",
        "criminal_justice", "education_scoring",
        "border_control", "critical_infrastructure",
        "law_enforcement", "autonomous_driving"
    ]
    LIMITED_KEYWORDS = [
        "chatbot", "emotion_recognition", "deepfake",
        "biometric_categorization"
    ]

    def classify(self, proposal: UseCaseProposal) -> RiskGrade:
        tags = set(proposal.data_sources + [proposal.ai_type])
        if any(k in tags for k in self.PROHIBITED_KEYWORDS):
            return RiskGrade.PROHIBITED
        if any(k in tags for k in self.HIGH_RISK_KEYWORDS):
            return RiskGrade.HIGH
        if proposal.is_generative:
            return RiskGrade.LIMITED
        if any(k in tags for k in self.LIMITED_KEYWORDS):
            return RiskGrade.LIMITED
        return RiskGrade.MINIMAL

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class UseCaseRegistry:
    """Central registry for all AI use case proposals."""

    def __init__(self):
        self._proposals: Dict[str, UseCaseProposal] = {}
        self._classifier = RiskClassifier()

    def submit(self, proposal: UseCaseProposal) -> UseCaseProposal:
        proposal.risk_grade = self._classifier.classify(proposal)
        proposal.status = ProposalStatus.SUBMITTED
        proposal.updated_at = datetime.now()
        self._proposals[proposal.id] = proposal

        if proposal.risk_grade == RiskGrade.PROHIBITED:
            proposal.status = ProposalStatus.REJECTED
        elif proposal.risk_grade in (RiskGrade.HIGH, RiskGrade.LIMITED):
            proposal.status = ProposalStatus.SCREENING
        else:
            proposal.status = ProposalStatus.APPROVED
        return proposal

    def screen_ethics(
        self, proposal_id: str, screening: EthicsScreening
    ) -> UseCaseProposal:
        p = self._proposals[proposal_id]
        p.ethics_screening = screening
        if screening.has_blockers():
            p.status = ProposalStatus.ON_HOLD
        else:
            p.status = ProposalStatus.COMMITTEE_REVIEW
        p.updated_at = datetime.now()
        return p

    def get_by_status(self, status: ProposalStatus) -> List[UseCaseProposal]:
        return [p for p in self._proposals.values() if p.status == status]

    def get_dashboard_stats(self) -> Dict[str, int]:
        stats = {}
        for s in ProposalStatus:
            stats[s.value] = len(self.get_by_status(s))
        return stats

**심의 프로세스 자동화 시스템(Committee Review Automation)**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum
from datetime import datetime, timedelta

class ReviewOutcome(Enum):
    APPROVED = "APPROVED"
    CONDITIONAL = "CONDITIONAL"
    REJECTED = "REJECTED"
    ON_HOLD = "ON_HOLD"

class AppealStatus(Enum):
    NONE = "NONE"
    FILED = "FILED"
    UNDER_REVIEW = "UNDER_REVIEW"
    UPHELD = "UPHELD"       # original decision maintained
    OVERTURNED = "OVERTURNED"

@dataclass
class ReviewScore:
    technical: float = 0.0      # max 30
    business: float = 0.0       # max 35
    ethics: float = 0.0         # max 20
    legal: float = 0.0          # max 15

    @property
    def total(self) -> float:
        return self.technical + self.business + self.ethics + self.legal

@dataclass
class CommitteeReview:
    proposal_id: str
    reviewer_scores: Dict[str, ReviewScore] = field(default_factory=dict)
    outcome: Optional[ReviewOutcome] = None
    conditions: List[str] = field(default_factory=list)
    rejection_reasons: List[str] = field(default_factory=list)
    appeal_status: AppealStatus = AppealStatus.NONE
    review_date: Optional[datetime] = None
    appeal_deadline: Optional[datetime] = None

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class CommitteeReviewEngine:
    """Automates committee review scoring and decision logic."""

    APPROVAL_THRESHOLD = 70
    CONDITIONAL_THRESHOLD = 55
    QUORUM_RATIO = 0.5
    SUPERMAJORITY_RATIO = 2 / 3
    APPEAL_WINDOW_DAYS = 10
    APPEAL_REVIEW_DAYS = 15

    def __init__(self, total_members: int = 7):
        self.total_members = total_members
        self._reviews: Dict[str, CommitteeReview] = {}

    def submit_score(
        self, proposal_id: str, reviewer_id: str, score: ReviewScore
    ) -> None:
        if proposal_id not in self._reviews:
            self._reviews[proposal_id] = CommitteeReview(
                proposal_id=proposal_id
            )
        self._reviews[proposal_id].reviewer_scores[reviewer_id] = score

    def finalize_review(self, proposal_id: str) -> CommitteeReview:
        review = self._reviews[proposal_id]
        scores = review.reviewer_scores

        # Check quorum
        if len(scores) < self.total_members * self.QUORUM_RATIO:
            review.outcome = ReviewOutcome.ON_HOLD
            review.conditions.append("Quorum not met - reschedule")
            return review

        # Calculate average score
        avg = sum(s.total for s in scores.values()) / len(scores)

        # Count approval votes (score >= CONDITIONAL_THRESHOLD)
        approvals = sum(
            1 for s in scores.values()
            if s.total >= self.CONDITIONAL_THRESHOLD
        )
        approval_rate = approvals / len(scores)

        # Check legal blocker
        legal_avg = sum(s.legal for s in scores.values()) / len(scores)
        has_legal_block = legal_avg < 5  # out of 15

        review.review_date = datetime.now()
        review.appeal_deadline = (
            datetime.now() + timedelta(days=self.APPEAL_WINDOW_DAYS)
        )

        if has_legal_block:
            review.outcome = ReviewOutcome.REJECTED
            review.rejection_reasons.append("Legal compliance score below minimum")
        elif avg >= self.APPROVAL_THRESHOLD and approval_rate >= self.SUPERMAJORITY_RATIO:
            review.outcome = ReviewOutcome.APPROVED
        elif avg >= self.CONDITIONAL_THRESHOLD:
            review.outcome = ReviewOutcome.CONDITIONAL
            review.conditions.append("Address identified gaps within 30 days")
        else:
            review.outcome = ReviewOutcome.REJECTED
            review.rejection_reasons.append(
                f"Average score {avg:.1f} below threshold"
            )

        return review

    def file_appeal(
        self, proposal_id: str, grounds: str
    ) -> CommitteeReview:
        review = self._reviews[proposal_id]
        if review.outcome != ReviewOutcome.REJECTED:
            raise ValueError("Appeals only for rejected proposals")
        if datetime.now() > review.appeal_deadline:
            raise ValueError("Appeal window has closed")
        review.appeal_status = AppealStatus.FILED
        return review

    def resolve_appeal(
        self, proposal_id: str, overturn: bool
    ) -> CommitteeReview:
        review = self._reviews[proposal_id]
        if overturn:
            review.appeal_status = AppealStatus.OVERTURNED
            review.outcome = ReviewOutcome.CONDITIONAL
        else:
            review.appeal_status = AppealStatus.UPHELD
        return review

**AI 포트폴리오 최적화 엔진(Portfolio Optimization Engine)**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from enum import Enum

class ProjectPhase(Enum):
    IDEATION = "IDEATION"
    POC = "POC"
    DEVELOPMENT = "DEVELOPMENT"
    DEPLOYMENT = "DEPLOYMENT"
    PRODUCTION = "PRODUCTION"
    RETIRED = "RETIRED"

class InvestmentCategory(Enum):
    CORE = "CORE"               # 70% budget
    ADJACENT = "ADJACENT"       # 20% budget
    TRANSFORMATIONAL = "TRANSFORMATIONAL"  # 10% budget

@dataclass
class AIProject:
    id: str
    title: str
    phase: ProjectPhase
    category: InvestmentCategory
    business_value: float       # 0-100
    strategic_fit: float        # 0-100
    technical_feasibility: float  # 0-100
    risk_level: float           # 0-100
    urgency: float              # 0-100
    synergy: float              # 0-100
    required_budget: float      # in currency units
    required_headcount: float   # FTE
    current_roi: float = 0.0

@dataclass
class PortfolioConstraints:
    total_budget: float
    total_headcount: float
    max_high_risk_projects: int = 3
    budget_allocation: Dict[str, float] = field(default_factory=lambda: {
        'CORE': 0.70, 'ADJACENT': 0.20, 'TRANSFORMATIONAL': 0.10
    })

@dataclass
class AIAssetInventory:
    """Central inventory for all AI assets."""
    asset_id: str
    system_name: str
    version: str
    owner_department: str
    risk_grade: int  # 1-4
    model_type: str
    data_sources: List[str]
    has_pii: bool
    deployment_env: str
    performance_metrics: Dict[str, float]
    last_audit_date: str
    next_review_date: str

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class PortfolioOptimizer:
    """MCDA-based portfolio optimization with resource constraints."""

    WEIGHTS = {
        'business_value': 0.25,
        'strategic_fit': 0.20,
        'technical_feasibility': 0.20,
        'risk_level': 0.15,
        'urgency': 0.10,
        'synergy': 0.10,
    }

    def __init__(self, constraints: PortfolioConstraints):
        self.constraints = constraints

    def calculate_mcda_score(self, project: AIProject) -> float:
        """Calculate weighted MCDA score for a project."""
        risk_inverted = 100 - project.risk_level
        raw = (
            project.business_value * self.WEIGHTS['business_value']
            + project.strategic_fit * self.WEIGHTS['strategic_fit']
            + project.technical_feasibility * self.WEIGHTS['technical_feasibility']
            + risk_inverted * self.WEIGHTS['risk_level']
            + project.urgency * self.WEIGHTS['urgency']
            + project.synergy * self.WEIGHTS['synergy']
        )
        return round(raw, 2)

    def optimize(
        self, projects: List[AIProject]
    ) -> Tuple[List[AIProject], Dict]:
        """Select optimal portfolio under resource constraints."""
        # Score all projects
        scored = [
            (self.calculate_mcda_score(p), p) for p in projects
        ]
        scored.sort(key=lambda x: x[0], reverse=True)

        selected = []
        remaining_budget = self.constraints.total_budget
        remaining_headcount = self.constraints.total_headcount
        category_budgets = {
            k: self.constraints.total_budget * v
            for k, v in self.constraints.budget_allocation.items()
        }
        high_risk_count = 0

        for score, project in scored:
            cat = project.category.value
            # Check constraints
            if project.required_budget > remaining_budget:
                continue
            if project.required_budget > category_budgets.get(cat, 0):
                continue
            if project.required_headcount > remaining_headcount:
                continue
            if project.risk_level > 70:
                if high_risk_count >= self.constraints.max_high_risk_projects:
                    continue
                high_risk_count += 1

            selected.append(project)
            remaining_budget -= project.required_budget
            remaining_headcount -= project.required_headcount
            category_budgets[cat] -= project.required_budget

        summary = {
            'total_selected': len(selected),
            'budget_utilized': self.constraints.total_budget - remaining_budget,
            'budget_remaining': remaining_budget,
            'headcount_utilized': self.constraints.total_headcount - remaining_headcount,
            'avg_mcda_score': (
                sum(self.calculate_mcda_score(p) for p in selected)
                / max(len(selected), 1)
            ),
        }
        return selected, summary

    def get_quadrant(self, project: AIProject) -> str:
        """Classify project into portfolio quadrant."""
        high_value = project.business_value >= 60
        high_feasibility = project.technical_feasibility >= 60
        if high_value and high_feasibility:
            return "Quick Wins"
        elif high_value and not high_feasibility:
            return "Big Bets"
        elif not high_value and high_feasibility:
            return "Low Hanging Fruits"
        else:
            return "Drop"

    def generate_status_report(
        self, projects: List[AIProject]
    ) -> Dict[str, List[str]]:
        """Generate portfolio status report by phase."""
        report: Dict[str, List[str]] = {}
        for p in projects:
            phase = p.phase.value
            if phase not in report:
                report[phase] = []
            report[phase].append(
                f"{p.id}: {p.title} (MCDA={self.calculate_mcda_score(p)})"
            )
        return report

**Shadow AI 탐지 시스템(Shadow AI Detection System)**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set, Optional
from datetime import datetime
from enum import Enum

class ShadowAIType(Enum):
    EXTERNAL_SAAS = "EXTERNAL_SAAS"
    UNAUTHORIZED_API = "UNAUTHORIZED_API"
    SELF_BUILT_MODEL = "SELF_BUILT_MODEL"
    RPA_AI_HYBRID = "RPA_AI_HYBRID"
    SPREADSHEET_ML = "SPREADSHEET_ML"

class DetectionSeverity(Enum):
    CRITICAL = "CRITICAL"   # PII or confidential data exposure
    HIGH = "HIGH"           # Unauthorized API integration
    MEDIUM = "MEDIUM"       # External SaaS usage
    LOW = "LOW"             # Spreadsheet-level ML

class FormalizationStatus(Enum):
    DETECTED = "DETECTED"
    UNDER_REVIEW = "UNDER_REVIEW"
    APPROVED_FOR_FORMALIZATION = "APPROVED"
    BLOCKED = "BLOCKED"
    FORMALIZED = "FORMALIZED"

@dataclass
class ShadowAIIncident:
    id: str
    detected_at: datetime
    ai_type: ShadowAIType
    severity: DetectionSeverity
    department: str
    description: str
    data_exposure_risk: bool = False
    formalization_status: FormalizationStatus = FormalizationStatus.DETECTED
    detection_method: str = ""

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class ShadowAIDetector:
    """Enterprise Shadow AI detection and management system."""

    # Known AI service domains to monitor
    KNOWN_AI_DOMAINS: Set[str] = {
        "api.openai.com", "chat.openai.com",
        "bard.google.com", "generativelanguage.googleapis.com",
        "api.anthropic.com", "api.cohere.ai",
        "huggingface.co", "api.replicate.com",
    }

    # ML libraries that indicate self-built models
    ML_LIBRARIES: Set[str] = {
        "tensorflow", "pytorch", "torch",
        "scikit-learn", "sklearn", "xgboost",
        "lightgbm", "keras", "transformers",
    }

    def __init__(self):
        self._incidents: List[ShadowAIIncident] = []
        self._whitelisted_domains: Set[str] = set()
        self._incident_counter = 0

    def whitelist_domain(self, domain: str) -> None:
        """Add an approved AI service domain."""
        self._whitelisted_domains.add(domain)

    def scan_network_logs(
        self, dns_logs: List[Dict]
    ) -> List[ShadowAIIncident]:
        """Scan DNS/network logs for unauthorized AI service access."""
        incidents = []
        for log in dns_logs:
            domain = log.get('destination_domain', '')
            if (domain in self.KNOWN_AI_DOMAINS
                    and domain not in self._whitelisted_domains):
                self._incident_counter += 1
                incident = ShadowAIIncident(
                    id=f"SAI-{self._incident_counter:04d}",
                    detected_at=datetime.now(),
                    ai_type=ShadowAIType.EXTERNAL_SAAS,
                    severity=DetectionSeverity.MEDIUM,
                    department=log.get('source_department', 'Unknown'),
                    description=f"Unauthorized access to {domain}",
                    detection_method="network_monitoring",
                )
                # Escalate if data upload detected
                if log.get('bytes_uploaded', 0) > 1_000_000:
                    incident.severity = DetectionSeverity.HIGH
                    incident.data_exposure_risk = True
                incidents.append(incident)
                self._incidents.append(incident)
        return incidents

    def scan_api_gateway(
        self, api_logs: List[Dict]
    ) -> List[ShadowAIIncident]:
        """Scan API gateway logs for unauthorized AI API integrations."""
        incidents = []
        for log in api_logs:
            endpoint = log.get('endpoint', '')
            if any(d in endpoint for d in self.KNOWN_AI_DOMAINS):
                if endpoint not in self._whitelisted_domains:
                    self._incident_counter += 1
                    incident = ShadowAIIncident(
                        id=f"SAI-{self._incident_counter:04d}",
                        detected_at=datetime.now(),
                        ai_type=ShadowAIType.UNAUTHORIZED_API,
                        severity=DetectionSeverity.HIGH,
                        department=log.get('department', 'Unknown'),
                        description=f"Unauthorized API call to {endpoint}",
                        detection_method="api_gateway",
                    )
                    incidents.append(incident)
                    self._incidents.append(incident)
        return incidents

    def scan_installed_packages(
        self, host_packages: Dict[str, List[str]]
    ) -> List[ShadowAIIncident]:
        """Scan hosts for unauthorized ML library installations."""
        incidents = []
        for host, packages in host_packages.items():
            found_ml = [p for p in packages if p in self.ML_LIBRARIES]
            if found_ml:
                self._incident_counter += 1
                incident = ShadowAIIncident(
                    id=f"SAI-{self._incident_counter:04d}",
                    detected_at=datetime.now(),
                    ai_type=ShadowAIType.SELF_BUILT_MODEL,
                    severity=DetectionSeverity.MEDIUM,
                    department="Unknown",
                    description=(
                        f"ML libraries on {host}: {', '.join(found_ml)}"
                    ),
                    detection_method="asset_scan",
                )
                incidents.append(incident)
                self._incidents.append(incident)
        return incidents

    def get_dashboard(self) -> Dict:
        """Generate Shadow AI monitoring dashboard data."""
        by_severity = {}
        by_type = {}
        by_dept = {}
        for inc in self._incidents:
            sev = inc.severity.value
            by_severity[sev] = by_severity.get(sev, 0) + 1
            typ = inc.ai_type.value
            by_type[typ] = by_type.get(typ, 0) + 1
            dept = inc.department
            by_dept[dept] = by_dept.get(dept, 0) + 1
        return {
            'total_incidents': len(self._incidents),
            'by_severity': by_severity,
            'by_type': by_type,
            'by_department': by_dept,
            'data_exposure_count': sum(
                1 for i in self._incidents if i.data_exposure_risk
            ),
        }

    def initiate_formalization(
        self, incident_id: str
    ) -> Optional[ShadowAIIncident]:
        """Begin formalization process for a Shadow AI incident."""
        for inc in self._incidents:
            if inc.id == incident_id:
                inc.formalization_status = (
                    FormalizationStatus.UNDER_REVIEW
                )
                return inc
        return None